# Wav2Sleep — SHHS Sleep Staging\n",
    "\n",
    "Trains the [wav2sleep](https://arxiv.org/abs/2411.04644) model on the Sleep Heart Health Study (SHHS) dataset using ECG, abdominal (ABD), and thoracic (THX) cardiorespiratory signals.\n",
    "\n",
    "## Experimental Setup\n",
    "- **Dataset**: SHHS polysomnography (ECG + ABD + THX signals, 30-second epochs)\n",
    "- **Task**: 4-class sleep staging — Wake, Light (N1+N2), Deep (N3), REM\n",
    "- **Split**: 70% train / 15% val / 15% test, split by patient to prevent data leakage\n",
    "- **Optimizer**: AdamW, lr=1e-3, weight_decay=1e-2, gradient clipping=1.0\n",
    "- **Early stopping**: patience=5 epochs on val accuracy\n",
    "\n",
    "## Ablation Studies\n",
    "1. **Feature dimension** (`feature_dim`): Tests 32 / 64 / 128 / 256 — controls model capacity\n",
    "2. **Learning rate**: Tests 1e-4 / 5e-4 / 1e-3 / 5e-3 — controls optimisation speed\n",
    "3. **Cross-modal generalisation**: Evaluates the trained model with ECG-only, respiratory-only, and all-modality subsets\n",
    "\n",
    "**Before running:**\n",
    "1. Paste the contents of `wav2sleep.py` into Cell 3\n",
    "2. Set `SHHS_ROOT` in Cell 4 to your SHHS data path, or leave as `None` to run on synthetic data

In [ ]:
# Cell 1 — Install dependencies
!pip install git+https://github.com/sunlabuiuc/PyHealth.git mne scipy --quiet

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/shhs')

In [ ]:
# Cell 3 — Paste wav2sleep.py contents here
# Copy everything from pyhealth/models/wav2sleep.py and paste below.
# This defines Wav2Sleep, SignalEncoder, ResidualBlock1D, and load_shhs_samples.

# === PASTE wav2sleep.py BELOW THIS LINE ===

In [ ]:
# Cell 4 — Configuration (edit before running)

# Path to SHHS polysomnography root on Google Drive.
# Set to None to run on synthetic data instead (always works without real data).
# Expected layout:
#   SHHS_ROOT/edfs/shhs1/*.edf
#   SHHS_ROOT/annotations-events-nsrr/shhs1/*-nsrr.xml
SHHS_ROOT = "/content/shhs/MyDrive/shhs/polysomnography"  # or None

# Number of recordings to load. Set to None for all.
MAX_RECORDINGS = 100

# Label scheme:
#   4-class (wav2sleep paper): merges N1+N2 as Light
#   5-class AASM             : {0:0, 1:1, 2:2, 3:3, 4:3, 5:4}
LABEL_MAP = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3}
NUM_CLASSES = len(set(LABEL_MAP.values()))

# Signal resolution (paper defaults: ecg=1024, resp=256)
ECG_SAMPLES  = 1024
RESP_SAMPLES = 256

# Patient-level split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

# Training hyper-parameters
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 1e-3
WEIGHT_DECAY  = 1e-2
PATIENCE      = 5
MAX_GRAD_NORM = 1.0

OUTPUT_DIR = "/content/wav2sleep_output"
SEED = 42

In [ ]:
# Cell 5 — Imports
import os
import random
from collections import Counter

import numpy as np
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.trainer import Trainer

# Wav2Sleep and load_shhs_samples are defined in Cell 3 (pasted from wav2sleep.py)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Cell 6 — Load data (SHHS or synthetic fallback)
#
# If SHHS_ROOT is set and valid, loads real EDF recordings.
# Otherwise falls back to synthetic data so the notebook always runs end-to-end.

INPUT_SCHEMA  = {"ecg": "tensor", "abd": "tensor", "thx": "tensor"}
OUTPUT_SCHEMA = {"label": "multiclass"}

def build_synthetic_samples(n_patients=10, epochs_per_patient=40, seed=SEED):
    rng = np.random.RandomState(seed)
    return [
        {
            "patient_id": f"patient-{pid:03d}",
            "visit_id":   f"epoch-{ep:04d}",
            "ecg":   rng.randn(1, ECG_SAMPLES).astype(np.float32),
            "abd":   rng.randn(1, RESP_SAMPLES).astype(np.float32),
            "thx":   rng.randn(1, RESP_SAMPLES).astype(np.float32),
            "label": rng.randint(0, NUM_CLASSES),
        }
        for pid in range(n_patients)
        for ep in range(epochs_per_patient)
    ]

if SHHS_ROOT and os.path.isdir(SHHS_ROOT):
    print(f"Loading SHHS recordings from {SHHS_ROOT} ...")
    samples = load_shhs_samples(
        shhs_root=SHHS_ROOT,
        epoch_seconds=30,
        ecg_samples_per_epoch=ECG_SAMPLES,
        resp_samples_per_epoch=RESP_SAMPLES,
        max_recordings=MAX_RECORDINGS,
        label_map=LABEL_MAP,
    )
    print(f"Loaded {len(samples):,} real epochs.")
else:
    print("SHHS_ROOT not set or unavailable — using synthetic data.")
    samples = build_synthetic_samples()
    print(f"Generated {len(samples):,} synthetic epochs.")

print(f"Label dist. : {dict(sorted(Counter(s['label'] for s in samples).items()))}")

In [ ]:
# Cell 7 — Patient-level train / val / test split
#
# Splitting by patient prevents the same person's epochs from
# appearing in both train and eval, which would inflate metrics.

patient_ids = list({s["patient_id"] for s in samples})
random.shuffle(patient_ids)

n_train = int(len(patient_ids) * TRAIN_RATIO)
n_val   = int(len(patient_ids) * VAL_RATIO)

train_pids = set(patient_ids[:n_train])
val_pids   = set(patient_ids[n_train: n_train + n_val])
test_pids  = set(patient_ids[n_train + n_val:])

train_samples = [s for s in samples if s["patient_id"] in train_pids]
val_samples   = [s for s in samples if s["patient_id"] in val_pids]
test_samples  = [s for s in samples if s["patient_id"] in test_pids]

print(f"Patients — train: {len(train_pids)}  val: {len(val_pids)}  test: {len(test_pids)}")
print(f"Epochs   — train: {len(train_samples):,}  val: {len(val_samples):,}  test: {len(test_samples):,}")

In [ ]:
# Cell 8 — Build SampleDatasets and DataLoaders
INPUT_SCHEMA  = {"ecg": "tensor", "abd": "tensor", "thx": "tensor"}
OUTPUT_SCHEMA = {"label": "multiclass"}

train_dataset = create_sample_dataset(train_samples, INPUT_SCHEMA, OUTPUT_SCHEMA, "shhs_train")
val_dataset   = create_sample_dataset(val_samples,   INPUT_SCHEMA, OUTPUT_SCHEMA, "shhs_val")
test_dataset  = create_sample_dataset(test_samples,  INPUT_SCHEMA, OUTPUT_SCHEMA, "shhs_test")

train_loader = get_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = get_dataloader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = get_dataloader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready.")

In [ ]:
# Cell 9 — Build model
model = Wav2Sleep(
    dataset=train_dataset,
    feature_dim=128,
    n_transformer_layers=2,
    n_attention_heads=8,
    transformer_ff_dim=512,
    dropout=0.1,
)
print(f"Wav2Sleep | params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 10 — Train
trainer = Trainer(
    model=model,
    metrics=["accuracy", "f1_weighted", "cohen_kappa"],
    output_path=OUTPUT_DIR,
    exp_name="wav2sleep_default",
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    epochs=EPOCHS,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": LR},
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    monitor="accuracy",
    monitor_criterion="max",
    load_best_model_at_last=True,
    patience=PATIENCE,
)

In [ ]:
# Cell 11 — Final test-set evaluation
test_scores = trainer.evaluate(test_loader)
print("\n=== Test Results ===")
for k, v in test_scores.items():
    print(f"  {k:<20s}: {v:.4f}")

In [ ]:
# Cell 12 — Cross-modal evaluation
#
# Same trained model, different modality subsets at inference.
# Demonstrates wav2sleep's ability to handle missing signals.

def eval_modality_subset(model, loader, keep_keys):
    model.eval()
    loss_total, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            filtered = {k: v for k, v in batch.items() if k in keep_keys or k == "label"}
            out = model(**filtered)
            loss_total += out["loss"].item() * out["y_true"].shape[0]
            correct += (out["y_prob"].argmax(-1) == out["y_true"]).sum().item()
            total += out["y_true"].shape[0]
    return loss_total / total, correct / total

subsets = {
    "ECG + ABD + THX": ["ecg", "abd", "thx"],
    "ECG only"        : ["ecg"],
    "ABD + THX only"  : ["abd", "thx"],
    "ECG + THX only"  : ["ecg", "thx"],
}

print("=== Cross-modal generalisation (test set) ===")
print(f"{'Modalities':<22}  {'loss':>8}  {'acc':>8}")
for name, keys in subsets.items():
    loss, acc = eval_modality_subset(model, test_loader, keys)
    print(f"{name:<22}  {loss:>8.4f}  {acc:>8.3f}")

In [ ]:
# Cell 13 — Ablation: feature_dim
#
# Tests four feature_dim settings and records val accuracy + F1.
# Copy the printed results into the docstring table in
# examples/shhs_sleep_staging_wav2sleep.py.

ablation_configs = [
    (32,  4),
    (64,  4),
    (128, 8),   # paper default
    (256, 8),
]

print(f"{'feature_dim':<12}  {'n_heads':>7}  {'#params':>10}  {'val_acc':>8}  {'val_f1':>8}")

for fdim, n_heads in ablation_configs:
    abl_model = Wav2Sleep(
        dataset=train_dataset,
        feature_dim=fdim,
        n_transformer_layers=2,
        n_attention_heads=n_heads,
        dropout=0.1,
    )
    abl_trainer = Trainer(
        model=abl_model,
        metrics=["accuracy", "f1_weighted"],
        output_path=OUTPUT_DIR,
        exp_name=f"wav2sleep_fdim{fdim}",
    )
    abl_trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=EPOCHS,
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": LR},
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        monitor="accuracy",
        monitor_criterion="max",
        load_best_model_at_last=True,
        patience=PATIENCE,
    )
    scores = abl_trainer.evaluate(val_loader)
    n_params = sum(p.numel() for p in abl_model.parameters())
    marker = "  <- paper default" if fdim == 128 else ""
    print(
        f"{fdim:<12}  {n_heads:>7}  {n_params:>10,}  "
        f"{scores.get('accuracy', 0):>8.3f}  "
        f"{scores.get('f1_weighted', 0):>8.3f}"
        f"{marker}"
    )

In [ ]:
# Cell 14 — Ablation: learning rate
#
# Keeps feature_dim=128 (paper default) fixed and varies the learning rate.
# This tests sensitivity to optimisation speed and whether 1e-3 is well-chosen.

lr_configs = [1e-4, 5e-4, 1e-3, 5e-3]

print(f"{'lr':<10}  {'val_acc':>8}  {'val_f1':>8}")

for lr in lr_configs:
    lr_model = Wav2Sleep(
        dataset=train_dataset,
        feature_dim=128,
        n_transformer_layers=2,
        n_attention_heads=8,
        dropout=0.1,
    )
    lr_trainer = Trainer(
        model=lr_model,
        metrics=["accuracy", "f1_weighted"],
        output_path=OUTPUT_DIR,
        exp_name=f"wav2sleep_lr{lr}",
    )
    lr_trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=EPOCHS,
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": lr},
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        monitor="accuracy",
        monitor_criterion="max",
        load_best_model_at_last=True,
        patience=PATIENCE,
    )
    scores = lr_trainer.evaluate(val_loader)
    marker = "  <- default" if lr == 1e-3 else ""
    print(
        f"{lr:<10}  "
        f"{scores.get('accuracy', 0):>8.3f}  "
        f"{scores.get('f1_weighted', 0):>8.3f}"
        f"{marker}"
    )